# Notebook 07 — ANOVA Basics

**Module:** 03 — Statistics and Probability  
**Notebook:** 07 of 18  
**Prerequisites:** NB06  
**Time estimate:** 75 minutes

---
## Step 1 — Motivation

The t-test handles two groups. ANOVA (Analysis of Variance) handles three or more.
In biology, multi-group comparisons are the norm: expression across tissue types,
response to multiple drug doses, expression in three developmental stages. Running
multiple t-tests instead inflates the Type I error rate — ANOVA is the correct tool.

---
## Step 2 — Intuition

ANOVA asks: 'Is the variance between groups larger than the variance within groups?'
If the F-statistic (between / within) is large, at least one group mean is different.
ANOVA tells you *that* a difference exists; post-hoc tests (Tukey's HSD) tell you *which* groups differ.

---
## Step 3 — Biological Background

**Example: gene expression across three tissue types (liver, brain, muscle)**
- $H_0$: all three tissues have the same mean expression for gene X
- $H_1$: at least one tissue differs

**ANOVA assumptions:**
1. Independence: samples are independent across and within groups
2. Normality: data within each group is approximately Normal
3. Homoscedasticity: equal variances across groups (test with Levene's test)

When assumption 3 fails: Welch's ANOVA (`scipy.stats.f_oneway` assumes equal variance;
use `pingouin.welch_anova` for unequal variance).

---
## Step 4 — Mathematical Explanation

**One-way ANOVA partitions total variance:**
$$SS_{\text{total}} = SS_{\text{between}} + SS_{\text{within}}$$

$$SS_{\text{between}} = \sum_{j=1}^k n_j(\bar{x}_j - \bar{x})^2$$
$$SS_{\text{within}} = \sum_{j=1}^k \sum_{i=1}^{n_j} (x_{ij} - \bar{x}_j)^2$$

**Mean squares:**
$$MS_{\text{between}} = SS_{\text{between}} / (k - 1)$$
$$MS_{\text{within}} = SS_{\text{within}} / (N - k)$$

**F-statistic:**
$$F = \frac{MS_{\text{between}}}{MS_{\text{within}}} \sim F(k-1,\; N-k) \text{ under } H_0$$

Under $H_0$: F ≈ 1. Under $H_1$: F >> 1.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt

In [ ]:
# Cell 6.1 — One-way ANOVA from scratch
def one_way_anova(*groups) -> dict:
    """
    One-way ANOVA for k groups.

    Parameters
    ----------
    *groups : arrays — one per group

    Returns
    -------
    dict with F, p_value, df_between, df_within, SS_between, SS_within
    """
    groups = [np.asarray(g, dtype=float) for g in groups]
    k = len(groups)
    N = sum(len(g) for g in groups)
    grand_mean = np.concatenate(groups).mean()

    SS_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    SS_within  = sum(np.sum((g - g.mean())**2) for g in groups)

    df_between = k - 1
    df_within  = N - k
    MS_between = SS_between / df_between
    MS_within  = SS_within  / df_within

    F = MS_between / MS_within
    p_value = stats.f.sf(F, df_between, df_within)

    return dict(F=F, p_value=p_value, df_between=df_between,
                df_within=df_within, SS_between=SS_between, SS_within=SS_within)

# Tissue expression example: liver, brain, muscle
rng = np.random.default_rng(42)
liver  = rng.normal(7.0, 1.0, 15)
brain  = rng.normal(8.5, 1.2, 15)
muscle = rng.normal(7.0, 1.0, 15)

res = one_way_anova(liver, brain, muscle)
sp_F, sp_p = stats.f_oneway(liver, brain, muscle)

print(f"Custom ANOVA: F={res['F']:.4f}, p={res['p_value']:.4f}")
print(f"scipy ANOVA:  F={sp_F:.4f}, p={sp_p:.4f}")
print(f"\nSS partition: SS_between={res['SS_between']:.2f}, SS_within={res['SS_within']:.2f}")

In [ ]:
# Cell 6.2 — Post-hoc: Tukey's HSD (which groups differ?)
# Use scipy.stats.tukey_hsd (available since scipy 1.8)
tukey = stats.tukey_hsd(liver, brain, muscle)
group_names = ["Liver", "Brain", "Muscle"]

print("Tukey's HSD post-hoc test (pairwise comparisons):")
for i in range(3):
    for j in range(i+1, 3):
        print(f"  {group_names[i]} vs. {group_names[j]}: p = {tukey.pvalue[i, j]:.4f}")

In [ ]:
# Cell 6.3 — Why not run multiple t-tests? Type I error inflation demo
rng2 = np.random.default_rng(0)
alpha = 0.05
n_sim = 10_000
k_groups = 5  # 5 groups = C(5,2) = 10 pairwise t-tests

anova_type1, ttests_type1 = 0, 0
for _ in range(n_sim):
    groups = [rng2.normal(0, 1, 10) for _ in range(k_groups)]
    # ANOVA
    _, p_anova = stats.f_oneway(*groups)
    if p_anova < alpha:
        anova_type1 += 1
    # All pairwise t-tests: flag if ANY is significant
    any_sig = False
    for i in range(k_groups):
        for j in range(i+1, k_groups):
            _, p_t = stats.ttest_ind(groups[i], groups[j])
            if p_t < alpha:
                any_sig = True
    if any_sig:
        ttests_type1 += 1

print(f"With k={k_groups} groups (H0 always true, N={n_sim:,} simulations):")
print(f"  ANOVA Type I rate:       {anova_type1/n_sim:.3f}  (nominal α = {alpha})")
print(f"  Multiple t-tests rate:   {ttests_type1/n_sim:.3f}  (expected: 1-(0.95)^10 = {1-0.95**10:.3f})")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — ANOVA visualization: box plots + F-distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
data = [liver, brain, muscle]
bp = ax.boxplot(data, labels=group_names, patch_artist=True)
colors_bp = ["steelblue", "tomato", "green"]
for patch, color in zip(bp['boxes'], colors_bp):
    patch.set_facecolor(color); patch.set_alpha(0.6)
ax.axhline(np.concatenate(data).mean(), color="black", ls="--", lw=1, label="Grand mean")
ax.set_ylabel("Expression (log₂CPM)")
ax.set_title(f"One-way ANOVA: F={res['F']:.2f}, p={res['p_value']:.4f}")
ax.legend()

ax = axes[1]
f_range = np.linspace(0, 8, 400)
ax.plot(f_range, stats.f.pdf(f_range, res['df_between'], res['df_within']),
        color="black", lw=2)
crit_f = stats.f.ppf(0.95, res['df_between'], res['df_within'])
ax.fill_between(f_range, stats.f.pdf(f_range, res['df_between'], res['df_within']),
                where=f_range > crit_f, color="tomato", alpha=0.5, label="p < 0.05 region")
ax.axvline(res['F'], color="steelblue", lw=2, label=f"Observed F = {res['F']:.2f}")
ax.set_xlabel("F"); ax.set_ylabel("Density")
ax.set_title(f"F-distribution (df={res['df_between']}, {res['df_within']})")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `one_way_anova()` from scratch. Verify it on your own three-group dataset.
2. Four drug doses (0, 10, 50, 100 mg) each tested on 12 cell cultures. Simulate
   expression data where only dose 100 differs from the others. Run ANOVA, then Tukey
   HSD. Does Tukey correctly identify dose 100 as different?
3. Test the homoscedasticity assumption for the tissue data using `scipy.stats.levene`.
   What does this tell you about whether ANOVA is appropriate?
4. Explain why running 10 pairwise t-tests inflates the Type I error rate from 5% to
   approximately 40%. Derive the formula $1 - (1-\alpha)^m$ and explain why it applies.

---
## Quiz — Active Recall

1. What does ANOVA test? Why can't you use multiple t-tests instead?
2. Write the F-statistic formula from memory.
3. What is SS_between and SS_within? What does each measure?
4. What does a post-hoc test add to ANOVA? Name one.
5. What is the homoscedasticity assumption in ANOVA? Why does it matter?

---
## Reflection

**Date completed:** ____________________

> *[Can you explain why ANOVA prevents Type I error inflation when multiple t-tests would not?]*

---
*Next: `08_nonparametric_tests.ipynb`*